# 4. Train
1. train set 준비
2. cluster center 초기화
3. NetVLAD 파라미터 초기화
4. Dataset 구성 (Query 기준)
5. triplet mining
6. loss 계산
7. optimizer step
8. validation


In [1]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
from collections import namedtuple
import random

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
import torch.autograd as Variable


import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


In [3]:
import sklearn
from sklearn.neighbors import NearestNeighbors

In [4]:
root_dir = './data/Pittsburgh250k/'
struct_dir = os.path.join(root_dir, 'netvlad_v100_datasets/datasets/')
queries_dir = os.path.join(root_dir, 'queries_real/')

In [5]:
def parse_dbStruct(structfile, dbPath):

    structfile
    dataset = structfile  #db의 이름을 넣기 위한 위치

    mat = loadmat(os.path.join(dbPath,structfile))

    matStruct = mat['dbStruct'].item()

    #debugging 용 출력
    print(len(matStruct))
    first_col = list(map(lambda x: x[0], matStruct))
    for i in range(len(matStruct)):
        print(f"matStruct[{i}] :{first_col[i]}")

    whichSet = matStruct[0].item()

    dbImage = [f[0].item() for f in matStruct[1]]  #이미지리스트
    utmDb = matStruct[2].T

    qImage = [f[0].item() for f in matStruct[3]] #쿼리 이미지
    utmQ = matStruct[4].T

    numDb = matStruct[5].item()
    numQ = matStruct[6].item()

    posDistThr = matStruct[7].item()  #25
    posDistSqThr = matStruct[8].item() #625 --> 25^2
    nonTrivPosDistSqThr = matStruct[9].item() #100 -->10^2

    return dbStruct(whichSet, dataset, dbImage, utmDb, qImage, 
        utmQ, numDb, numQ, posDistThr, 
        posDistSqThr, nonTrivPosDistSqThr)

dbStruct = namedtuple('dbStruct', ['whichSet', 'dataset', 
    'dbImage', 'utmDb', 'qImage', 'utmQ', 'numDb', 'numQ',
    'posDistThr', 'posDistSqThr', 'nonTrivPosDistSqThr'])

train = parse_dbStruct('pitts30k_train.mat',struct_dir)

def input_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
    ])

class WholeDatset(data.Dataset):

    def __init__(self, dbPath, stPath, qPath, structFile, transform=None, onlyDB=False):
        super().__init__()  #parent class 초기화용이나, 현재는 크게필요하지 않음. 
        self.input_transform = transform #tensor로 변환
        self.dbStruct = parse_dbStruct(structFile, stPath) #dataset에 대한 파일 읽기

        self.images = [os.path.join(dbPath, dbIm) for dbIm in self.dbStruct.dbImage]
        if not onlyDB:
            self.images += [os.path.join(qPath, qIm) for qIm in self.dbStruct.qImage]

        self.whichSet = self.dbStruct.whichSet  #train, test, val 중 하나
        self.dataset = self.dbStruct.dataset   # pittsburgh250k, 30k 등

        self.positives = None   #현재는 없음
        self.distances = None   #현재는 없음

    def __len__(self):
            return len(self.images)

    def __getitem__(self, index):
        img = Image.open(self.images[index])  #dataset의 이미지를 불러와 출력

        if self.input_transform:
            img = self.input_transform(img)  #tensor로 변환한다. 
        return img, index

    def getPositive(self):   #학습에선 사용하지 않음. 이후 Test/Evaluation에서 GT추출용으로 사용 
        
        #Data의 숫자가 크지 않아 sklearn으로 아직까지 가능할 듯.         
        if  self.positives is None:
            knn = NearestNeighbors(n_jobs=-1)
            knn.fit(self.dbStruct.utmDb)

            self.distances, self.positives = knn.radius_neighbors(self.dbStruct.utmQ,
                    radius=self.dbStruct.posDistThr)

        return self.positives

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]


In [6]:
whole_train_set = WholeDatset(
    dbPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_train.mat',
    transform=input_transform(),
    onlyDB=False
    )


whole_val_set = WholeDatset(
    dbPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_val.mat',
    transform=input_transform(),
    onlyDB=False
    )

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]
10
matStruct[0] :val
matStruct[1] :[array(['000/000000_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585200.91088968 585200.91088968 585200.91088968 ... 584439.93767933
 584439.93767933 584439.93767933]
matStruct[3] :[array(['000/000015_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585001.41335051 585001.41335051 585001.41335051 ... 584534.69627173
 584534.69627173 584534.69627173]
matStruct[5] :[10000]
matStruct[6] :[7608]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]


In [ ]:
# from torch.utils.data import DataLoader, SubsetRandomSampler


# def getCluster(encoder, dataset, num_clusters=64, num_images=100, desc_per_image=500, batch_size=4, device='cuda'):
#     encoder.eval() 

#     loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
#     #Parameter Initialize
#     #테스트용 코드 작성과는 다르게, Ref.코드에서는 이미지 100개에서, 5만개의 Desciptor를 샘플링한다. 
#     #따라서, 데이터셋에서 100개의 이미지를 랜덤샘플하고, 여기서 각각 500개의 descriptor를 추출해서 Clustering을 진행한다. 
#     #https://m.blog.naver.com/kwangrok21/222412219800 SubsetRandomSampler 사용법법

#     from math import ceil
#     from torch.utils.data import SubsetRandomSampler
#     import numpy as np
#     from sklearn.cluster import KMeans
    

#     nDesrciptor = 50000
#     nPerImages = 100
#     nIm = ceil(nDesrciptor / nPerImages)
#     sampler = SubsetRandomSampler(np.random.choice(len(datasets), nIm, replace=False))

#     data_loader = DataLoader(datasets, sampler=sampler)

#     mymodel.eval()
#     desc_list = []

#     print(data_loader)
#     count = 0 
#     with torch.no_grad():
#         for batch in data_loader:      #data_loader는 DataLoader object이지만, CNN입력은 tensor(B,C,H,W)이어야 하므로, loop로 돌려야 한다. 
#             if isinstance(batch, (list, tuple)):
#                 x = batch[0]          # 보통 image, 지금 이쪽으로 들어온다. 
#             else:
#                 x = batch
#             count = count + 1
            
#             print(f"x.type is {x.dtype} and {count}, batch shape : {batch[0].shape}")

#             x = x.to(device)
#             feat = mymodel(x)
#             B,C,H,W = feat.shape
#             feat = feat.permute(0,2,3,1).reshape(-1,C)

#             idx = np.random.choice(feat.shape[0], nPerImages, replace=False)
#             feat = feat[idx]

#             desc_list.append(feat.cpu().numpy())

#     X_np = np.concatenate(desc_list, axis=0)

#     print("clustering")
#     kmeans = KMeans(n_clusters=K, random_state=0, n_init=10)
#     kmeans.fit(X_np)

#     centroids = kmeans.cluster_centers_

#     return centroids
    
    
    # 이건 getCluster사용하기 전에 먼저 할 것. 
    # x = x.to(device)

In [26]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans

def getCluster(encoder, dataset, num_clusters=64, num_images=100, desc_per_image=500, batch_size=4, device='cuda'):
    encoder.eval()

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    all_descs = []
    img_count = 0

    with torch.no_grad():
        for batch in loader:
            # dataset이 이미지 텐서만 반환한다고 가정
            if isinstance(batch, (list, tuple)):
                x = batch[0]
            else:
                x = batch

            x = x.to(device)
            feat = encoder(x)                     # (B, C, H, W)
            B, C, H, W = feat.shape

            # (B, C, H, W) -> (B, H*W, C)
            feat = feat.permute(0, 2, 3, 1).reshape(B, H * W, C)

            for b in range(B):
                desc = feat[b]                   # (H*W, C)

                # 이미지당 descriptor 일부만 샘플링
                n_sample = min(desc_per_image, desc.shape[0])
                idx = torch.randperm(desc.shape[0], device=desc.device)[:n_sample]
                sampled = desc[idx]              # (n_sample, C)

                all_descs.append(sampled.cpu().numpy())
                img_count += 1

                if img_count >= num_images:
                    break

            if img_count >= num_images:
                break

    # (M, C)
    all_descs = np.concatenate(all_descs, axis=0)

    # k-means
    kmeans = KMeans(n_clusters=num_clusters, random_state=0, n_init=10)
    kmeans.fit(all_descs)

    centroids = kmeans.cluster_centers_         # (K, C)

    return centroids, all_descs

In [8]:
class VGG16Feature(nn.Module):
    def __init__(self):
        super().__init__()
        
        #encoder = models.vgg16(pretrained=True) 버전이 바뀌면서 워닝이 뜬다.
        encoder = models.vgg16(weights="VGG16_Weights.IMAGENET1K_FEATURES")
        # capture only feature part and remove last relu and maxpool
        layers = list(encoder.features.children())[:-2]
        
        self.encoder = nn.Sequential(*layers)
        self.encoder_dim = 512

        for p in self.encoder.parameters():
            p.requires_grad = False
    
    def forward(self, x):
        x = self.encoder(x)
        return x
#Class 끝 

#데이터 준비

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
])


In [9]:
#cuda 확인
if torch.cuda.is_available() :
    device = 'cuda'
else :
    device = 'cpu' 

print(device)


cuda


In [ ]:
model = VGG16Feature().to(device)
# centroids = getCluster(model, whole_train_set)
centroids, train_descs = getCluster(
    model,
    whole_train_set,
    num_clusters=64,
    num_images=100,
    desc_per_image=500,
    batch_size=4,
    device=device
)

x.type is torch.float32 and 1, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 2, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 3, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 4, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 5, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 6, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 7, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 8, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 9, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 10, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 11, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 12, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 13, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 14, ba

In [11]:
print(model)

VGG16Feature(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, di

In [12]:

class NetVLAD(nn.Module):
    #VLAD Layer
    def __init__(self, num_clusters = 64, dim = 128,normalize_input = True):
    
    #        Args:
    # num_clusters : int
    #     The number of clusters
    # dim : int
    #     Dimension of descriptors
    # alpha : float
    #     Parameter of initialization. Larger value is harder assignment.
    # normalize_input : bool
    #     If true, descriptor-wise L2 normalization is applied to input.

        super(NetVLAD, self).__init__()
        self.num_clusters = num_clusters  #cluster개수를 정의해야 vlad vector를 정의할 수 있다. 
        self.dim = dim   #ref코드는 128로 초기화했는데, 만약 VGG-16을 쓴다면 512를 써야 한다. 
        self.alpha = 0  #soft-assignment를 위한 alpha값. scratch 구현에선 2~10정도로 임의 설정했으나, 이제 이걸 학습해나가야 한다. 
        
        self.normalize_input = normalize_input  #이게 필요한가..? 정규화 여부를 저장한다. bool.
        self.conv = nn.Conv2d(dim, num_clusters, kernel_size=(1, 1), bias=False)  #assign용 연산. 아마도 alpha값과 centroid의 벡터곱 등에 쓴다. 
        self.centroids = nn.Parameter(torch.rand(num_clusters, dim))
        #자리 만들기. nn.Parameter 함수를 써서 학습파라미터로 선언한다. none으로 설정하면 네트워크 설정하면서 optimizer를 붙일수 없다. 
        #그래서 뭐라도 넣어놔야 한다. 


    def init_params(self, centroids, descriptors):
        #실제로 cluster의 centroid와 descritor를 받는 부분
        #이 함수를 통해 c_k, alpha, conv.weight와 conv.bias를 통해 assignment score(z_k)를 계산한다. 
        #즉 soft-assignment 관련 항목을 초기화한다. 

        #해야 할것 
        # 1. centroid c_k를 학습할 수 있도록 파라미터로 등록
        # 2. assignment score z_k를 계산하는 self.conv(x)의 weight를 셋팅한다. 
            
            
            #타입 맞추기
            device = descriptors.device
            dtype = descriptors.dtype


            #getCluster를 연산하면 numpy형태로 반환받는다. 그걸 텐서로 바꾼다.
            #기존 scratch에서는 torch.tensor를 사용했고, 이번에는 torch.as_tensor를 사용한다. 
            # 참고자료 https://jh-bk.tistory.com/46
            
            centroids_t = torch.as_tensor(centroids, dtype=dtype, device=device)   # (K=64, dim=512)

            desc_norm = F.normalize(descriptors, p=2, dim=1)       # (M=B*H*W, C)
            cent_norm = F.normalize(centroids_t, p=2, dim=1)       # (K, C)

            # W^T = 2 * alpha * c_k.t() * x_i  

            dots = torch.matmul(cent_norm, desc_norm.t())          
            # (K, M)형태로 출력하기 위해 편의상 transpose의 위치가 바뀐다. 
            # 이는 ref. 코드에서도 동일하다. dots = np.dot(clstsAssign, traindescs.T)
            dots, _ = torch.sort(dots, dim=0, descending=True)

            self.alpha = (-torch.log(torch.tensor(0.01, device=device, dtype=dtype))
                        / torch.mean(dots[0, :] - dots[1, :])).item()

            self.centroids = nn.Parameter(centroids_t)

            self.conv.weight = nn.Parameter(
                (self.alpha * cent_norm).unsqueeze(-1).unsqueeze(-1)   # (K, C, 1, 1)
            )
            self.conv.bias = None

    def forward(self, x):
        N, C = x.shape[:2]

        if self.normalize_input:
            x = F.normalize(x, p=2, dim=1)   # (N, C, H, W)

        soft_assign = self.conv(x)                               # (N, K, H, W)
        soft_assign = soft_assign.view(N, self.num_clusters, -1) # (N, K, HW)
        soft_assign = F.softmax(soft_assign, dim=1)              # (N, K, HW)

        x_flatten = x.view(N, C, -1)                             # (N, C, HW)

        vlad = torch.zeros(
            N, self.num_clusters, C,
            dtype=x.dtype,
            device=x.device
        )                                                        # (N, K, C)

        for k in range(self.num_clusters):
            centroid = self.centroids[k].view(1, C, 1)           # (1, C, 1)
            residual = x_flatten - centroid                      # (N, C, HW)

            assign_weight = soft_assign[:, k, :].view(N, 1, -1)  # (N, 1, HW)
            residual = residual * assign_weight                  # (N, C, HW)

            vlad[:, k, :] = residual.sum(dim=2)                  # (N, C)

        vlad = F.normalize(vlad, p=2, dim=2)                     # (N, K, C)
        vlad = vlad.view(N, -1)                                  # (N, K*C)
        vlad = F.normalize(vlad, p=2, dim=1)                     # (N, K*C)

        return vlad

In [ ]:
# encoder = VGG16Feature()
# pool = NetVLAD(num_clusters=64, dim=512)

# print(pool)
# model = nn.Module()
# model.add_module('encoder', encoder)
# model.add_module('pool', pool)

# model.to(device)

# print(model)

NetVLAD(
  (conv): Conv2d(512, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
)
Module(
  (encoder): VGG16Feature(
    (encoder): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, k

# Training 을 위한 구성. 

## Training Dataset
  - Dataset형태로 선언(Class)
  - __init__(self)  
     - 데이터 선언: 기본데이터/mat파일로부터 데이터 파싱
     - Positive 분류, Negative 분류
  - __getitem__(self)  
    - query 이미지
    - positive 이미지
    - negative 이미지

  - __getlen__(self)  
    - query 이미지가 곧 길이가 된다다. 



In [14]:
class QueryDataset(data.Dataset):
    def __init__(self, rootPath, stPath, qPath, structFile, nNegSample = 1000, nNeg=10, margin = 0.1, input_transform = None):
        super().__init__()

        self.input_transform = input_transform
        self.margin = margin

        self.rootPath = rootPath
        self.stPath = stPath
        self.qPath = qPath
        self.structFile = structFile

        self.dbStruct = parse_dbStruct(structFile, stPath) #dataset에 대한 파일 읽기

        self.nNegSample = nNegSample # number of negatives to randomly sample
        self.nNeg = nNeg # number of negatives used for training

        #Test/Eval 시 WholeDataset 만으로도 수행 가능하도록 images도 선언함. (query + db) or(only db)
        self.db_images = [os.path.join(self.rootPath, dbIm) for dbIm in self.dbStruct.dbImage]
        self.q_images = [os.path.join(self.qPath, qIm) for qIm in self.dbStruct.qImage]

        self.positives = None   #현재는 없음
        self.distances = None   #현재는 없음


        #mat data load and parsing
        self.whichSet = self.dbStruct.whichSet              #train
        self.dataset = self.dbStruct.dataset                #pittsburch250k


        #Positive 계산을 위한 사전작업
        knn = NearestNeighbors(n_jobs=-1)
        knn.fit(self.dbStruct.utmDb)

        #거리 25m 이내의 positive 후보군들을 검색한다. 
        #기본 설정은 민코프스키 로 되어있으므로, 자동으로 유클리디안 거리가 나온다. 
        #여기서 의문은, trivial positive를 어떻게 제외하지 하는 부분이다. 

        PositiveDistance,PositiveIndex = knn.radius_neighbors(
            self.dbStruct.utmQ,
            radius=self.dbStruct.posDistThr,
            return_distance=True
            )

        nontrivial_positives = []

        self.triplets = [] #Hard negative 를 위한 추가변수

        for i in range(len(PositiveIndex)):
            fDist = PositiveDistance[i]
            nIndex = PositiveIndex[i]
            fDistSq = fDist ** 2
            mask = fDistSq > self.dbStruct.nonTrivPosDistSqThr
            #np masking 방법 https://m.blog.naver.com/baek2sm/221844619151
            nontrivial_positives.append(nIndex[mask])

        self.pos_within_Thr = PositiveIndex  #trivial 도 포함
        self.nontrivial_pos = nontrivial_positives 

        #예외처리를 위한 부분. 만약 nontrivial possitive가 없다면 해당 쿼리는 제외한다. 
        self.queries = np.where(np.array([len(x) for x in self.nontrivial_pos])>0)[0]

        # Negative 후보 만들기 
        self.potential_negatives = []        

        # Index로 연산하기 위해 전체 Index를 하나 만든다. 
        self.numDbIndex = np.arange(self.dbStruct.numDb)

        #전체 dbImage 배열에서 Potential Positve를 뺀 나머지 배열을 만든다. 
        for pos in self.pos_within_Thr:
            self.potential_negatives.append(
                                            np.setdiff1d(
                                                         self.numDbIndex, pos, 
                                                         assume_unique=True
                                                         )
                                            )
        # hard negative mining 결과 저장용
        self.triplets = []

    def getPositive(self):   #학습에선 사용하지 않음. 이후 Test/Evaluation에서 GT추출용으로 사용 
        return self.pos_within_Thr

    def getNegative(self):
        return self.potential_negatives

    def getNontrivialPositive(self):
        return self.nontrivial_pos
    
    def getValidQueries(self):
        return self.queries

    def __getitem__(self, idx):
        #모든 인덱스 다 받아 올것. triplet은 다른 곳에서 구한다. 
        #q_idx = self.queries[idx]
        q_idx, pos_idx, neg_idx = self.triplets[idx]

        #Hard negative 적용을 위한 변경, 랜덤 초이스를 제외
        # pos_idx = np.random.choice(self.nontrivial_pos[q_idx])
        # neg_idx = np.random.choice(self.potential_negatives[q_idx])

        q_img = Image.open(self.q_images[q_idx]).convert("RGB")
        p_img = Image.open(self.db_images[pos_idx]).convert("RGB")
        n_img = Image.open(self.db_images[neg_idx]).convert("RGB")

        if self.input_transform is not None:
            q_img = self.input_transform(q_img)
            p_img = self.input_transform(p_img)
            n_img = self.input_transform(n_img)

        return q_img, p_img, n_img, q_idx, pos_idx, neg_idx

    def __len__(self):
        return len(self.queries)


query_train_set = QueryDataset(
    rootPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_train.mat',
    input_transform = input_transform()
    )

print(query_train_set.getPositive())

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]
[array([  0,  17,  18,  19,  20,  21,  22,  23,  16,  15,   1,   2,   3,
          4,   5,   6,   7,   8,   9,  10,  11,  12,  35,  34,  33,  24,
         25,  26,  27,  28,  29,  30,  32,  31,  13,  14, 124, 120, 121,
        122, 123, 126, 125, 135, 127, 128, 129, 130, 134, 131, 132, 133,
        137, 136, 140, 138, 139, 141, 143, 142,  72,  83,  82,  81,  80,
         79,  78,  77,  76,  75,  74,  84,  73,  87,  86,  85,  88,  89,
         90,  91,  92,  95,  94,  93,  41,  36,  42,  43,  44,  45, 

In [16]:
loader = DataLoader(query_train_set, batch_size=2, shuffle=True)

# q, p, n, q_idx, p_idx, n_idx = next(iter(loader))  #디버그를 위한 q,p,n idx 출력 받음.

# print(q.shape)
# print(p.shape)
# print(n.shape)

In [ ]:
class EmbedNet(nn.Module):
    def __init__(self, encoder, pool):
        super().__init__()
        self.encoder = encoder
        self.pool = pool

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        return x



encoder = VGG16Feature().to(device)

centroids, train_descs = getCluster(
    encoder,
    whole_train_set,
    num_clusters=64,
    num_images=100,
    desc_per_image=500,
    batch_size=4,
    device=device
)

pool = NetVLAD(num_clusters=64, dim=512).to(device)
pool.init_params(centroids, train_descs)

model = EmbedNet(encoder, pool).to(device)

In [ ]:
#임시 비활성화
# train_loader = DataLoader(
#     query_train_set,
#     batch_size=4,
#     shuffle=True,
#     num_workers=0
# )

In [18]:
criterion = nn.TripletMarginLoss(margin=0.1, p=2)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer = torch.optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.001
)
#scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

# TensorBoard + Train/Val

아래 셀은 다음을 수행한다.

1. epoch마다 `train loss` 기록  
2. epoch마다 validation set에서 `Recall@1/5/10` 계산  
3. TensorBoard에 scalar 기록  
4. best `Recall@1` 모델 저장

> 주의: validation 전체 descriptor 추출은 시간이 오래 걸릴 수 있다.


In [19]:
from torch.utils.tensorboard import SummaryWriter
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader


In [20]:
class DbImageDataset(data.Dataset):
    def __init__(self, image_paths, input_transform=None):
        self.image_paths = image_paths
        self.input_transform = input_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.input_transform is not None:
            img = self.input_transform(img)
        return img, idx


class QueryImageDataset(data.Dataset):
    def __init__(self, image_paths, input_transform=None):
        self.image_paths = image_paths
        self.input_transform = input_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.input_transform is not None:
            img = self.input_transform(img)
        return img, idx


def compute_db_q_descriptors(model, query_set, device, batch_size=4, num_workers=0):
    model.eval()

    db_loader = DataLoader(
        DbImageDataset(query_set.db_images, query_set.input_transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    q_loader = DataLoader(
        QueryImageDataset(query_set.q_images, query_set.input_transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    db_descs = []
    q_descs = []

    with torch.no_grad():
        for images, _ in db_loader:
            images = images.to(device)
            desc = model(images)
            desc = F.normalize(desc, p=2, dim=1)
            db_descs.append(desc.cpu())

        for images, _ in q_loader:
            images = images.to(device)
            desc = model(images)
            desc = F.normalize(desc, p=2, dim=1)
            q_descs.append(desc.cpu())

    db_descs = torch.cat(db_descs, dim=0).numpy()
    q_descs = torch.cat(q_descs, dim=0).numpy()

    return db_descs, q_descs

In [21]:
def build_hard_triplets(query_set, model, device, batch_size=4, num_workers=0):
    db_descs, q_descs = compute_db_q_descriptors(
        model=model,
        query_set=query_set,
        device=device,
        batch_size=batch_size,
        num_workers=num_workers
    )

    triplets = []

    for q_idx in query_set.queries:
        pos_candidates = query_set.nontrivial_pos[q_idx]
        if len(pos_candidates) == 0:
            continue

        pos_idx = np.random.choice(pos_candidates)

        neg_pool = query_set.potential_negatives[q_idx]
        if len(neg_pool) == 0:
            continue

        n_sample = min(query_set.nNegSample, len(neg_pool))
        neg_candidates = np.random.choice(neg_pool, size=n_sample, replace=False)

        q_desc = q_descs[q_idx]
        neg_desc = db_descs[neg_candidates]

        d_neg = np.linalg.norm(neg_desc - q_desc[None, :], axis=1)
        sorted_idx = np.argsort(d_neg)
        topk = min(10, len(sorted_idx))
        chosen = np.random.choice(sorted_idx[:topk])
        neg_idx = neg_candidates[chosen]

        triplets.append((q_idx, pos_idx, neg_idx))

    query_set.triplets = triplets
    print(f"Built {len(triplets)} hard triplets")

In [22]:
def evaluate_recall(model, whole_set, device, batch_size=4, num_workers=0, topk=(1, 5, 10)):
    model.eval()

    loader = DataLoader(
        whole_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    all_desc = []


    with torch.no_grad():
        for images, indices in loader:
            images = images.to(device)
            desc = model.pool(model.encoder(images))
            all_desc.append(desc.cpu())  #CPU 에서 연산을 위한 이동. sklearn이 CPU에서 돌아간다. 만약 FAISS라면 GPU연산이 가능하다. 
            #all_indices.append(indices)

    all_desc = torch.cat(all_desc, dim=0)

    n_db = whole_set.dbStruct.numDb
    n_q = whole_set.dbStruct.numQ

    db_desc = all_desc[:n_db].numpy()
    q_desc = all_desc[n_db:n_db + n_q].numpy()

    knn = NearestNeighbors(
        n_neighbors=max(topk),
        metric='euclidean',
        n_jobs=-1
    )
    knn.fit(db_desc)
    distances, indices = knn.kneighbors(q_desc)

    utmDb = whole_set.dbStruct.utmDb
    utmQ = whole_set.dbStruct.utmQ
    posDistThr = whole_set.dbStruct.posDistThr

    knn_gt = NearestNeighbors(n_jobs=-1)
    knn_gt.fit(utmDb)
    positives = knn_gt.radius_neighbors(
        utmQ,
        radius=posDistThr,
        return_distance=False
    )

    recalls = {}
    for k in topk:
        correct = 0
        for i in range(len(q_desc)):
            pred = indices[i, :k]
            gt = positives[i]
            if np.intersect1d(pred, gt).size > 0:
                correct += 1
        recalls[k] = correct / len(q_desc)

    return recalls


In [23]:
def train_one_epoch(model, loader, optimizer, criterion, device, writer=None, epoch=0):
    model.train()
    running_loss = 0.0
    
    #Batch loop
    for batch_idx, (q, p, n, q_idx, pos_idx, neg_idx) in enumerate(loader):
        q = q.to(device)
        p = p.to(device)
        n = n.to(device)

        optimizer.zero_grad()

        q_desc = model(q)
        p_desc = model(p)
        n_desc = model(n)

        # ---- debug ----
        utmQ = query_train_set.dbStruct.utmQ
        utmDb = query_train_set.dbStruct.utmDb

        if (batch_idx % 100) == 0:
            for b in range(len(q_idx)):
                qp = np.linalg.norm(utmQ[q_idx[b].item()] - utmDb[pos_idx[b].item()])
                qn = np.linalg.norm(utmQ[q_idx[b].item()] - utmDb[neg_idx[b].item()])
                print("gps q-p:", qp, "gps q-n:", qn)        
            q0 = q_desc[0].detach().cpu().numpy()
            p0 = p_desc[0].detach().cpu().numpy()
            n0 = n_desc[0].detach().cpu().numpy()
            positive_feature_distance = np.linalg.norm(q0 - p0)
            negative_feature_distance = np.linalg.norm(q0 - n0)
            print("criterion:", criterion)
            print("query_idx:", q_idx[0].item(), "utm:", np.array2string(utmQ[q_idx[0].item()], precision=4, separator=", "), "descriptor_shape:", q0.shape, "descriptor_head:", np.array2string(q0[:8], precision=4, separator=", "))
            print("positive_idx:", pos_idx[0].item(), "descriptor_shape:", p0.shape, "descriptor_head:", np.array2string(p0[:8], precision=4, separator=", "))
            print("positive_feature_distance_from_query:", positive_feature_distance)
            print("negative_idx:", neg_idx[0].item(), "descriptor_shape:", n0.shape, "descriptor_head:", np.array2string(n0[:8], precision=4, separator=", "))
            print("negative_feature_distance_from_query:", negative_feature_distance)
        # ---- debug end ----


        loss = criterion(q_desc, p_desc, n_desc)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()


        #TensorBoard용 코드
        global_step = epoch * len(loader) + batch_idx
        if writer is not None:
            writer.add_scalar("train/loss_iter", loss.item(), global_step)

        if batch_idx % 10 == 0:
            print(f"epoch {epoch+1} | batch {batch_idx}/{len(loader)} | loss = {loss.item():.4f}")



    avg_loss = running_loss / len(loader)

    if writer is not None:
        writer.add_scalar("train/loss_epoch", avg_loss, epoch)
        writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)

    return avg_loss


In [24]:
# TensorBoard 실행 전에 log_dir를 먼저 만든다.
log_dir = "runs/netvlad_trainval"
writer = SummaryWriter(log_dir=log_dir)

In [ ]:


# num_epochs = 20
# val_every = 1
# best_r1 = -1.0
# best_ckpt_path = "best_netvlad_model.pth"

# for epoch in range(num_epochs):
#     avg_loss = train_one_epoch(
#         model=model,
#         loader=train_loader,
#         optimizer=optimizer,
#         criterion=criterion,
#         device=device,
#         writer=writer,
#         epoch=epoch
#     )

#     print(f"[Epoch {epoch+1}] train loss = {avg_loss:.4f}")

#     if (epoch + 1) % val_every == 0:
#         recalls = evaluate_recall(
#             model=model,
#             whole_set=whole_val_set,
#             device=device,
#             batch_size=4,
#             num_workers=0,
#             topk=(1, 5, 10)
#         )
        
#         #TensorBoard용 코드
#         writer.add_scalar("val/Recall@1", recalls[1], epoch)
#         writer.add_scalar("val/Recall@5", recalls[5], epoch)
#         writer.add_scalar("val/Recall@10", recalls[10], epoch)

#         print(
#             f"[Epoch {epoch+1}] "
#             f"R@1={recalls[1]:.4f}, "
#             f"R@5={recalls[5]:.4f}, "
#             f"R@10={recalls[10]:.4f}"
#         )

#         if recalls[1] > best_r1:
#             best_r1 = recalls[1]
#             torch.save(model.state_dict(), best_ckpt_path)
#             print(f"best model updated -> {best_ckpt_path} (R@1={best_r1:.4f})")

# writer.close()
# print("training + validation finished")
# print(f"log_dir: {log_dir}")
# print(f"best Recall@1: {best_r1:.4f}")


In [25]:
num_epochs = 20
val_every = 1
best_r1 = -1.0
best_ckpt_path = "best_netvlad_model.pth"

for epoch in range(num_epochs):
    print(f"\n[Epoch {epoch+1}] Hard negative mining...")
    build_hard_triplets(
        query_set=query_train_set,
        model=model,
        device=device,
        batch_size=4,
        num_workers=0
    )

    train_loader = DataLoader(
        query_train_set,
        batch_size=4,
        shuffle=True,
        num_workers=0
    )

    avg_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        writer=writer,
        epoch=epoch
    )

    print(f"[Epoch {epoch+1}] train loss = {avg_loss:.4f}")

    if (epoch + 1) % val_every == 0:
        recalls = evaluate_recall(
            model=model,
            whole_set=whole_val_set,
            device=device,
            batch_size=4,
            num_workers=0,
            topk=(1, 5, 10)
        )

        writer.add_scalar("val/Recall@1", recalls[1], epoch)
        writer.add_scalar("val/Recall@5", recalls[5], epoch)
        writer.add_scalar("val/Recall@10", recalls[10], epoch)

        print(
            f"[Epoch {epoch+1}] "
            f"R@1={recalls[1]:.4f}, "
            f"R@5={recalls[5]:.4f}, "
            f"R@10={recalls[10]:.4f}"
        )

        if recalls[1] > best_r1:
            best_r1 = recalls[1]
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"best model updated -> {best_ckpt_path} (R@1={best_r1:.4f})")

writer.close()
print("training + validation finished")
print(f"log_dir: {log_dir}")
print(f"best Recall@1: {best_r1:.4f}")


[Epoch 1] Hard negative mining...
Built 7416 hard triplets
gps q-p: 14.26212498876454 gps q-n: 128.5774766485955
gps q-p: 13.321119821361433 gps q-n: 163.23511426473962
gps q-p: 24.547532815980286 gps q-n: 120.10893064571952
gps q-p: 18.434641012160256 gps q-n: 183.3883420889214
criterion: TripletMarginLoss()
query_idx: 6938 utm: [ 584773.5265, 4477457.1878] descriptor_shape: (32768,) descriptor_head: [-0.0046, -0.0041, -0.0086, -0.006 , -0.0082, -0.0059, -0.0026, -0.0047]
positive_idx: 3556 descriptor_shape: (32768,) descriptor_head: [-0.0047, -0.0041, -0.0087, -0.0062, -0.0082, -0.0059, -0.0025, -0.0047]
positive_feature_distance_from_query: 0.019764574
negative_idx: 863 descriptor_shape: (32768,) descriptor_head: [-0.0046, -0.0041, -0.0086, -0.0061, -0.0082, -0.0059, -0.0026, -0.0047]
negative_feature_distance_from_query: 0.009869507
epoch 1 | batch 0/1854 | loss = 0.1065
epoch 1 | batch 10/1854 | loss = 0.1077
epoch 1 | batch 20/1854 | loss = 0.1068
epoch 1 | batch 30/1854 | loss 

KeyboardInterrupt: 